# Transformer segmentation of vulnerable road users in adverse conditions

Ayman Lakhnati, MAAI901. This notebook reproduces the primary HRDA versus MIC(HRDA) comparison on all **406 original ACDC validation images** and runs a separate supervised DINOv3 experiment. It reads actual files and writes measured artifacts. No output cells contain invented results.

Primary question: does MIC improve the minimum condition-specific mean IoU for **person, rider and bicycle** by at least 2.0 percentage points, while no condition loses more than 1.0 point? Fog, nighttime, rain and snow remain separate. The weakest condition can differ between models.

The primary comparison reuses published pretrained/adapted checkpoints. The separate DINOv3 extension trains a 1x1 semantic head on labeled ACDC training images with a frozen recent transformer. **That extension is supervised target-domain training.** It does not isolate MIC or reproduce unsupervised adaptation training. Read [topic requirements](../docs/topic_requirements.md) and the repository README before running.

## Environment

Run from the repository root or `notebooks/`, using the project Python environment. Install matching CUDA PyTorch wheels using the official PyTorch instructions before installing requirements if needed. Install the pinned requirements and restart the kernel after changing PyTorch. GPU execution is strongly recommended for full-resolution sliding-window inference. The CPU path can take much longer.

Images and external checkpoints retain their original licenses. Downloads need internet access and substantial disk space. Keep raw images and model weights outside GitHub.
On Colab or Kaggle, select a GPU runtime before running cells. Cloud execution defaults to a separate clean rerun checkout pinned to frozen source-only commit `2b4a5325f7b5843a6d5e5f1d52922a05f926c9c7`. The final main submission will include measured reference evidence and can contain nonempty training/result directories without prediction files; it is not the default clean rerun workspace. The setup cell prints the actual commit and branch/ref before installing dependencies. Restart the kernel if PyTorch changes.

Locally, an existing project checkout stays in place and may reuse completed artifacts only after the later provenance checks. Set `FRESH_RERUN = True` for a separate pinned clean rerun. Existing files are never deleted, moved or automatically reset. If the chosen rerun directory already contains an incompatible checkout or an interrupted experiment, choose a new `RERUN_DIRECTORY` and preserve the previous evidence. Identical commands do not guarantee identical bits across devices or platforms.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, HERE.parent) if (p/'roadseg'/'metrics.py').is_file()), None)
ON_CLOUD = 'COLAB_RELEASE_TAG' in os.environ or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
EXPERIMENT_COMMIT = '2b4a5325f7b5843a6d5e5f1d52922a05f926c9c7'
REPOSITORY_URL = 'https://github.com/AymanLakhnati/Adverse-Condition-VRU-Segmentation.git'
FRESH_RERUN = ON_CLOUD or ROOT is None  # Locally set True for a separate pinned rerun.
RERUN_DIRECTORY = (ROOT.parent if ROOT is not None else HERE)/'vru-segmentation-rerun-2b4a532'

def git_value(*arguments, cwd):
    result = subprocess.run(['git', *arguments], cwd=cwd, capture_output=True, text=True)
    return result.stdout.strip() if result.returncode == 0 else None

if FRESH_RERUN:
    destination = RERUN_DIRECTORY.resolve()
    if not destination.exists():
        subprocess.run(['git','clone','--no-checkout',REPOSITORY_URL,str(destination)], check=True)
        # Checkout only the newly created clone, never the user's existing workspace.
        subprocess.run(['git','checkout','--detach',EXPERIMENT_COMMIT], cwd=destination, check=True)
    else:
        assert git_value('rev-parse','--verify','HEAD',cwd=destination) == EXPERIMENT_COMMIT, (
            'Rerun directory exists at another ref. Preserve it and choose a new RERUN_DIRECTORY.')
        assert not git_value('status','--porcelain','--untracked-files=no',cwd=destination), (
            'Rerun checkout has tracked changes. Preserve them and choose a new RERUN_DIRECTORY.')
    ROOT = destination
    assert git_value('rev-parse','--verify','HEAD',cwd=ROOT) == EXPERIMENT_COMMIT
else:
    assert ROOT is not None, 'No current project checkout found. Set FRESH_RERUN = True.'
assert (ROOT/'roadseg'/'metrics.py').is_file(), 'Repository source unavailable.'
os.chdir(ROOT)

def run(*arguments):
    command = [sys.executable, *map(str, arguments)]
    print(' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)
print('Mode:', 'pinned separate rerun' if FRESH_RERUN else 'reuse current local checkout')
print('Repository:', ROOT)
print('Actual commit:', git_value('rev-parse','--verify','HEAD',cwd=ROOT) or '(no committed HEAD)')
print('Actual branch/ref:', git_value('branch','--show-current',cwd=ROOT) or '(detached or uncommitted HEAD)')
print('Pinned clean-rerun commit:', EXPERIMENT_COMMIT)
print('Kernel:', sys.executable)


In [ ]:
INSTALL = ON_CLOUD  # Set True in another fresh project environment.
if INSTALL:
    run('-m', 'pip', 'install', '-r', 'requirements.txt')
import torch
import pandas as pd
from IPython.display import display, Image, Markdown
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, 'device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))
DATA = ROOT/'data'/'acdc'
RESULTS = ROOT/'results'
HRDA = ROOT/'checkpoints/hrda/csHR2acdcHR_hrda_d3a68/iter_40000_relevant.pth'
MIC = ROOT/'checkpoints/mic/csHR2acdcHR_mic_hrda_b52f1/iter_40000_relevant.pth'


## Official data and published checkpoints

The data helper streams the official RGB ZIP and retains 1,600 training plus 406 validation images. The server does not support range requests, so selecting these members still transfers a large archive stream (about 15.6 GB). The helper checks CRC32 for retained members and records provenance. Ground-truth semantic masks must use `labelTrainIds`, not RGB palette illustrations or original `labelIds`. The checkpoint helper supplies published HRDA/MIC weights. The ACDC helper first downloads and MD5-verifies the semantic labels, then streams RGB images. Review official ACDC research terms and model terms before downloading.

The next cell skips complete existing RGB data. It can take considerable time on a new machine. Run checkpoint/label setup before inference. The explicit checks below prevent incomplete downloads from silently becoming a 406-image result.

In [ ]:
download_helper = ROOT/'roadseg'/'download_checkpoints.py'
if download_helper.is_file():
    run('-m', 'roadseg.download_checkpoints')
else:
    raise FileNotFoundError('Missing roadseg/download_checkpoints.py')
run('-m', 'roadseg.download_acdc', '--root', DATA, '--labels-only')
train_images = list(DATA.glob('rgb_anon/*/train/*/*_rgb_anon.png'))
val_images = list(DATA.glob('rgb_anon/*/val/*/*_rgb_anon.png'))
if len(train_images) != 1600 or len(val_images) != 406:
    run('-m', 'roadseg.download_acdc', '--root', DATA, '--include-train')


In [ ]:
expected = {'fog':100, 'night':106, 'rain':100, 'snow':100}
actual = {condition:len(list((DATA/'rgb_anon'/condition/'val').glob('*/*_rgb_anon.png'))) for condition in expected}
assert actual == expected, f'Incomplete original validation data: {actual}'
assert len(list(DATA.glob('rgb_anon/*/train/*/*_rgb_anon.png'))) == 1600
for checkpoint in (HRDA, MIC):
    assert checkpoint.is_file(), f'Missing published checkpoint: {checkpoint}'
val_masks = list(DATA.glob('gt/*/val/*/*_gt_labelTrainIds.png'))
train_masks = list(DATA.glob('gt/*/train/*/*_gt_labelTrainIds.png'))
assert len(val_masks) == 406, f'Expected 406 validation train-ID masks, found {len(val_masks)}'
assert len(train_masks) == 1600, f'Expected 1600 training train-ID masks, found {len(train_masks)}'
print('Data/checkpoint presence verified. Counts alone do not establish split provenance.')


## HRDA and MIC inference

The supplied checkpoints already contain their authors' adaptation training. This step performs our own inference and evaluation, preserving their 19-class prediction space. It uses deterministic float32 evaluation with official normalization and nested sliding grids, without flipping or multiscale augmentation. The implementation records hashes, device, runtime and per-image timing. Existing completed manifest entries allow resuming an interrupted run. Keep the same checkpoint and settings throughout a run.

The primary experiment does not use ACDC validation labels to fit or select models. ACDC training labels are reserved for the clearly separated supervised extension.

In [ ]:
run('-m', 'roadseg.run_inference', '--data', DATA, '--output', RESULTS,
    '--hrda', HRDA, '--mic', MIC, '--device', DEVICE, '--expected', '406')
manifest = pd.read_csv(RESULTS/'manifest.csv')
assert len(manifest) == 406 and manifest.image_id.nunique() == 406
assert manifest.condition.value_counts().to_dict() == expected
display(manifest.groupby('condition').size().rename('images').to_frame())


## Exact evaluation and the proposed hypothesis

The evaluator ignores only GT 255. It rejects invalid predictions on valid pixels and any mask shape mismatch. It sums confusion counts per condition before computing class IoU. Person, rider and bicycle are train IDs 11, 12 and 18. An absent union has undefined IoU, not perfect IoU. V(c) needs all three classes and W needs all four conditions. Missing evidence leaves the hypothesis unresolved.

The bootstrap resamples paired images within condition, then recomputes pooled counts and each model's minimum. These descriptive intervals assume independent images. They exclude uncertainty from source clustering, checkpoint training and domain shift.

In [ ]:
run('-m', 'roadseg.metrics', RESULTS/'manifest.csv', '--path-root', ROOT, '--output', RESULTS,
    '--bootstrap', '2000', '--seed', '42', '--require-official-validation')
summary = json.loads((RESULTS/'summary.json').read_text())
display(pd.read_csv(RESULTS/'condition_metrics.csv'))
display(summary['hypothesis'])
display(summary.get('bootstrap', {}))


In [ ]:
per_class = pd.read_csv(RESULTS/'per_class.csv')
vru = per_class[per_class.class_id.isin([11,12,18])]
display(vru[['model','condition','class_name','support','iou','precision','recall','f1']])
for model in ('hrda','mic'):
    print(model, 'W:', summary['models'][model]['W'], 'weakest condition(s):', summary['models'][model]['weakest_conditions'])


## Separate supervised DINOv3 experiment

DINOv3 (2025) supplies frozen dense patch features. A newly initialized 1x1 head maps the 384 feature channels to 19 semantic classes, with **7,315 trained parameters** (384 x 19 + 19). Before a fresh five-epoch fit on all 1,600 labeled training images, three one-epoch trials compare learning rates 0.0003, 0.001 and 0.003 using recording-family-disjoint internal fitting/development groups drawn only from those training images. Within each condition, matching GoPro chapters (for example GOPR0376 and GP010376) stay in one conservative recording family. A seed-42 ordered recording-family subset targets the nearest feasible 20% development frames; equal-distance count ties favor more development frames, then seeded subset order resolves equal counts; large families can cause a substantial deviation, so inspect the actual saved counts. Each trial resets the same head initialization and fitting crop/flip randomness. All trial weights are discarded before the fresh full refit. One deterministic 512-pixel center crop per development image defines pooled VRU IoU for selection, with lower learning rates winning exact ties. If every VRU score is undefined, the declared present-class mIoU fallback applies and the saved result explicitly identifies it. The 406 official validation images remain unused for tuning. This short-budget center-crop proxy is not exhaustive optimization. The encoder stays frozen.

This is an additional supervised target-domain experiment with a different supervision budget and architecture. Its results cannot identify MIC's contribution or support a fair ranking against unsupervised checkpoints. Keep the original HRDA/MIC hypothesis and results separate. The following module owns its exact recorded preprocessing, optimization and training configuration.

In [ ]:
for module in ('tune_dinov3.py', 'train_dinov3.py', 'evaluate_dinov3.py'):
    assert (ROOT/'roadseg'/module).is_file(), f'Missing supervised pipeline module: {module}'
# Main workflow: python -m roadseg.tune_dinov3 --device cuda --refit
# DEVICE also supports CPU when CUDA is unavailable.
run('-m', 'roadseg.tune_dinov3', '--help')


In [ ]:
# Train-only search and fresh full refit. Never tune on official validation/test.
DINO_TRAIN = ROOT/'outputs'/'dinov3_dense'
DINO_TUNING = ROOT/'outputs'/'dinov3_tuning'
tuning_path = DINO_TUNING/'tuning_results.json'
config_path = DINO_TRAIN/'training_config.json'
if not tuning_path.exists():
    run('-m', 'roadseg.tune_dinov3', '--data-root', DATA, '--output', DINO_TUNING,
        '--refit', '--refit-output', DINO_TRAIN, '--device', DEVICE, '--seed', '42')
tuning = json.loads(tuning_path.read_text())
assert tuning['status'] == 'completed' and len(tuning['trials']) == 3, (
    'Existing search is incomplete. Preserve it and choose a new tuning output directory before restarting.')
assert sorted(t['learning_rate'] for t in tuning['trials']) == [0.0003, 0.001, 0.003]
assert tuning['trial_epochs'] == 1 and tuning['seed'] == 42
assert all(t['fit_epochs'] == 1 and t['fit_optimizer_steps'] > 0 and t['encoder_unchanged']
           for t in tuning['trials'])
assert tuning['official_validation_or_test_used'] is False
assert tuning['split']['recording_families_disjoint'] is True
assert tuning['split']['sequence_groups_disjoint'] and tuning['split']['frame_ids_disjoint']
assert not set(tuning['split']['fit_groups']) & set(tuning['split']['development_groups'])
assert tuning['split']['fit_images'] + tuning['split']['development_images'] == 1600
selected_lr = tuning['selected_learning_rate']
print('Selected learning rate:', selected_lr, 'selection metric:', tuning['selected_metric'])
if tuning.get('selection_fallback'):
    print('Actual selection fallback:', tuning['selection_fallback'])
display(pd.read_csv(DINO_TUNING/'tuning_trials.csv'))
if not config_path.exists():
    # A completed search can launch a fresh refit without rerunning or reusing trial weights.
    run('-m', 'roadseg.train_dinov3', '--data-root', DATA, '--output', DINO_TRAIN,
        '--device', DEVICE, '--batch-size', '4', '--epochs', '5', '--seed', '42',
        '--learning-rate', selected_lr, '--tuning-results', tuning_path)
training_config = json.loads(config_path.read_text())
assert training_config.get('status') == 'completed' and training_config.get('completed_fixed_full_protocol'), (
    'Existing refit is incomplete or exploratory. Preserve it and choose a new output path before restarting.')
import hashlib
assert training_config['learning_rate'] == selected_lr
assert training_config['tuning_results_sha256'] == hashlib.sha256(tuning_path.read_bytes()).hexdigest()
assert len(tuning['all_training_content_sha256']) == 64
assert training_config['all_training_content_sha256'] == tuning['all_training_content_sha256']
assert training_config['official_validation_used_for_selection'] is False
assert training_config['internal_training_development_used_for_learning_rate_selection'] is True
assert training_config['head_before_sha256'] == tuning['initial_head_sha256']
assert training_config['encoder_state_sha256'] == tuning['encoder_state_sha256']
assert training_config['head_parameters'] == 7315
assert training_config['training_images'] == 1600 and training_config['available_train_images'] == 1600
assert training_config['completed_epochs'] == 5 and training_config['epochs'] == 5
assert training_config['batch_size'] == 4 and training_config['seed'] == 42
assert training_config['encoder_state_unchanged'] and training_config['head_state_changed']
print('Using completed tuned full refit:', DINO_TRAIN)
display(training_config)
run('-m', 'roadseg.evaluate_dinov3', '--head', DINO_TRAIN/'head.pt',
    '--data-root', DATA, '--output', RESULTS/'dinov3', '--device', DEVICE, '--batch-size', '4')
dino_results = json.loads((RESULTS/'dinov3'/'summary.json').read_text())
assert dino_results['validation_images'] == 406
display(pd.DataFrame([{'condition':c, **{k:v for k,v in values.items() if k != 'per_class'}}
                      for c,values in dino_results['conditions'].items()]))
print('Supervised DINOv3 W:', dino_results['W'])
print('Learning setting:', dino_results['learning_setting'])


## Figures and sample outputs

Figures use the saved measured artifacts. Public sample panels include only our own predicted segmentation masks and identify their source image IDs. Raw RGB photographs and ground-truth masks remain local and excluded from GitHub under ACDC terms. A private local viewer can inspect them after an authorized dataset download. Review condition-level VRU scores, the class-by-condition heatmap, confusion counts and the actual DINOv3 training curve together.

In [ ]:
if (ROOT/'roadseg'/'visualize.py').exists():
    run('-m', 'roadseg.visualize')
figure_names = ['condition_vru.png','class_condition_heatmap.png','sample_predictions.png',
                'confusion_vru.png','dinov3_training_curve.png']
for name in figure_names:
    path = RESULTS/'figures'/name
    if path.exists():
        display(Markdown(f'**{name}**'))
        display(Image(filename=str(path)))
    else:
        print('Figure not generated yet:', name)


## Submission checks

Report the measured hypothesis outcome, including an unsupported or unresolved result. Distinguish published training from our own checkpoint inference and fresh supervised DINOv3 head training. Use source citations for borrowed benchmark values and never present them as our measurements. Validation evaluation does not equal an official ACDC test-server result.

The technical report must stay within 2,500 words. Submit runnable code through GitHub, the report, figures and sample outputs. The presentation should use the same JSON results. Explain collection geography, small/occluded VRUs, pseudo-label bias and annotation uncertainty. These scores do not establish autonomous-driving safety. See [topic requirements](../docs/topic_requirements.md) for the document/rubric mapping.